# Perceptrons
You should build an end-to-end machine learning pipeline using a perceptron model. In particular, you should do the following:
- Load the `mnist` dataset using [Pandas](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html). You can find this dataset in the datasets folder.
- Split the dataset into training and test sets using [Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
- Build an end-to-end machine learning pipeline, including a [perceptron](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Perceptron.html) model.
- Optimize your pipeline by validating your design decisions.
- Test the best pipeline on the test set and report various [evaluation metrics](https://scikit-learn.org/0.15/modules/model_evaluation.html).  
- Check the documentation to identify the most important hyperparameters, attributes, and methods of the model. Use them in practice.

# Loading the Dataset

In [14]:
import pandas as pd
from sklearn.pipeline import Pipeline

from sklearn.linear_model import Perceptron

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split,GridSearchCV



In [15]:
df = pd.read_csv("https://raw.githubusercontent.com/m-mahdavi/teaching/refs/heads/main/datasets/mnist.csv")
df.drop("id", axis=1, inplace=True)
df.head()


,class,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


# Split to train and test set

In [22]:
X = df.drop("class", axis=1)
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% test data
    random_state=42,    # reproducibility
    stratify=y          # keeps digit distribution balanced
)
print("df size:", df.shape)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("trainset:", X_train_scaled.shape)
print("testset:", X_test_scaled.shape)

df size: (4000, 785)
trainset: (3200, 784)
testset: (800, 784)


#Perceprtron Training

In [17]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Perceptron())
])

#HyperParameter Tuning

In [18]:
param_grid = {
    "model__penalty": [None, "l2", "l1", "elasticnet"],
    "model__alpha": [0.0001, 0.001, 0.01],
    "model__max_iter": [1000, 2000],
    "model__eta0": [1.0, 0.1, 0.01]
}

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', Perceptron())]),
             n_jobs=-1,
             param_grid={'model__alpha': [0.0001, 0.001, 0.01],
                         'model__eta0': [1.0, 0.1, 0.01],
                         'model__max_iter': [1000, 2000],
                         'model__penalty': [None, 'l2', 'l1', 'elasticnet']},
             scoring='accuracy', verbose=2)

In [19]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Best Parameters: {'model__alpha': 0.001, 'model__eta0': 0.01, 'model__max_iter': 1000, 'model__penalty': 'l1'}
Best CV Score: 0.8512507524325478


#Train Best model

In [20]:
best_model = grid_search.best_estimator_

#Evaluate on Test Set

In [21]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8525

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.97      0.94        75
           1       0.93      0.93      0.93        97
           2       0.86      0.88      0.87        78
           3       0.83      0.82      0.83        84
           4       0.95      0.80      0.87        74
           5       0.78      0.73      0.75        73
           6       0.86      0.94      0.90        78
           7       0.85      0.91      0.88        85
           8       0.78      0.73      0.76        83
           9       0.77      0.79      0.78        73

    accuracy                           0.85       800
   macro avg       0.85      0.85      0.85       800
weighted avg       0.85      0.85      0.85       800


Confusion Matrix:
 [[73  0  0  1  0  1  0  0  0  0]
 [ 0 90  0  2  0  0  0  0  5  0]
 [ 1  0 69  1  0  0  3  2  1  1]
 [ 0  0  2 69  0  3  3  4  1  2]
 [ 1  0  1  0 59  0  1  4  3  5]
 [ 3  3  2  4 